# Walkthrough: Loading and Visualizing Extracted Objects
This Jupyter Notebook provides a step-by-step guide on how to load and visualize extracted objects. Follow the instructions below to understand the workflow.

## The construction pipeline
This repo hosts a pipeline for constructing semantic scenegraph based on RGBD images captured from Iphone. The pipeline can be executed by the following command
```bash
python build_map.py <path to RGBD images dataset>
```
The pipeline is as follow
1. Images, odometry, and camera intrinsics are loaded from the dataset
2. Segmentation, VLM, Captioning, and LLM models are loaded
3. Each images are segmented with Segmentation model
4. Fragments are cropped out and projected onto a 3d space
5. Crops are semantically embedded with Image Embedding model
6. Crops and Fragments are associated and merged with each other based on their semantic and geometric similarity
7. After all images are processed, the captioning and embedding models go through all objects and label them
8. Extract labels and edges based on image and semantic dat

## The object class (Model.tracker.MapObject)
The `MapObject` class is used to store information about objects. It includes the following attributes:
- `pcd`: `o3d.geometry.PointCloud` - The point cloud representation of the object.
- `bbox`: `o3d.geometry.AxisAlignedBoundingBox` - The bounding box of the object.
- `clip_ft`: `torch.Tensor` - The aggregated feature tensor (from image).
- `text_ft`: `torch.Tensor` - The aggregated feature tensor (from semantic label).
- `crops`: `List[tuple]` - A list of tuples containing crop information (from images) (`crop`, `xyxy`, `frame_idx`, `score`).
- `oid`: `Optional[int]` - The object ID (optional).
- `class_ids`: `List[int]` - A list of class IDs associated with the object.
- `class_name`: `str` - The name of the object's class.
- `detections`: `List[Detection]` - A list of detections associated with the object.
- `num_views`: `int` - The number of views in which the object is visible (default is 1).
- `total_weight`: `float` - The total weight of the object (default is 1.0).

## The tracker class (Model.tracker.ObjectTracker3D)
The `ObjectTracker3D` class is responsible for managing and associating objects in the dataset. It includes the following attributes:
- `objects`: `MapObjectList` - A list to store all the objects.
- `voxel_size`: `float` - The voxel size used for downsampling point clouds.
- `w_geo`: `float` - The weight for geometric similarity during object association.
- `w_sem`: `float` - The weight for semantic similarity during object association.
- `edges`: `List[tuple]` - A list of edges representing relationships between objects.
- `match_threshold`: `float` - The threshold for matching objects based on similarity.
- `max_points`: `int` - The maximum number of points allowed in the point cloud representation of an object.


In [1]:
from Model.tracker import *
from utils.logger import *
import open3d as o3d
from utils.io import *
from Model.models import *
from utils.data_construction.plot_graph import *
from utils.graph import *
import json
from VPR_im2im.utils.data_loader import *
%load_ext autoreload
%autoreload 2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
with open("graph_dataset/graph.json", 'rb') as f:
    g = json.load(f)

In [12]:
g['edges'][50]

{'src_id': 10,
 'dst_id': 22,
 'src_name': 'red office stool',
 'dst_name': 'red office chair',
 'relation': 'near'}

In [13]:
df = pd.read_csv('graph_dataset/raw_scans/iphone/3578aa5730/odometry.csv')

In [14]:
df.head()

,timestamp,frame,x,y,z,qx,qy,qz,qw
0,6321.641064,0,0.0,0.0,0.0,0.600268,-0.664773,0.322900,0.305762
1,6321.657732,1,0.0,0.0,0.0,0.600902,-0.665074,0.322655,0.304116
2,6321.674400,2,0.0,0.0,0.0,0.600666,-0.665195,0.322539,0.304441
3,6321.691068,3,0.0,0.0,0.0,0.600144,-0.665090,0.322510,0.305727
4,6321.707735,4,0.0,0.0,0.0,0.598922,-0.665001,0.322512,0.308305


In [2]:
# Initialize the logger for tracking and debugging
logger = build_logger()

# Load the full tracker from the specified checkpoint directory using the logger
tracker, _ = load_full_tracker("graph_dataset/raw_extration/checkpoints", logger)

[21:30:15] [INFO] [Resume] Loading tracker checkpoint from: graph_dataset/raw_extration/checkpoints


[21:30:15] [INFO] [Resume] Loaded object_0000
[21:30:15] [INFO] [Resume] Loaded object_0001
[21:30:15] [INFO] [Resume] Loaded object_0002
[21:30:15] [INFO] [Resume] Loaded object_0003
[21:30:15] [INFO] [Resume] Loaded object_0004
[21:30:15] [INFO] [Resume] Loaded object_0005
[21:30:15] [INFO] [Resume] Loaded object_0006
[21:30:15] [INFO] [Resume] Loaded object_0007
[21:30:15] [INFO] [Resume] Loaded object_0008
[21:30:15] [INFO] [Resume] Loaded object_0009
[21:30:15] [INFO] [Resume] Loaded object_0010
[21:30:15] [INFO] [Resume] Loaded object_0011
[21:30:15] [INFO] [Resume] Loaded object_0012
[21:30:15] [INFO] [Resume] Loaded object_0013
[21:30:15] [INFO] [Resume] Loaded object_0014
[21:30:15] [INFO] [Resume] Loaded object_0015
[21:30:15] [INFO] [Resume] Loaded object_0016
[21:30:15] [INFO] [Resume] Loaded object_0017
[21:30:15] [INFO] [Resume] Loaded object_0018
[21:30:15] [INFO] [Resume] Loaded object_0019
[21:30:15] [INFO] [Resume] Loaded object_0020
[21:30:15] [INFO] [Resume] Loaded 

## Room assignment
As of the moment, room assignment is done manually by human. The program will display an overlay of detected objects onto a floorplan generated by the SLAM process. Human will manually draw polygon and define room names.

In [3]:
# The method below will generate an html file
# After the file is generated, user will open it on browser and define rooms through it
plot_scene_graph_over_floorplan_manual(tracker, "graph_dataset/floorplan.png", outfile="graph_dataset/scene_graph.html")

[(0, 21, {'rtype': 'near', 'score': 0.21096969617076589, 'dist': 0.9336244810789158, 'src': 'white desktop printer', 'dst': 'research poster board'}), (0, 7, {'rtype': 'near', 'score': 0.06768075045367923, 'dist': 1.6157721122187716, 'src': 'white desktop printer', 'dst': 'wooden door handle'}), (0, 20, {'rtype': 'near', 'score': 0.11053802418474282, 'dist': 1.3214374461924714, 'src': 'white desktop printer', 'dst': 'wall posters display'}), (0, 1, {'rtype': 'near', 'score': 0.1287320429726476, 'dist': 1.2300133532337398, 'src': 'white desktop printer', 'dst': 'red office chair'}), (0, 19, {'rtype': 'near', 'score': 0.1398265047816346, 'dist': 1.1804117455373975, 'src': 'white desktop printer', 'dst': 'Poster on wall.'}), (1, 34, {'rtype': 'near', 'score': 0.12258124813295253, 'dist': 1.259388752261145, 'src': 'red office chair', 'dst': 'Large white board'}), (1, 37, {'rtype': 'near', 'score': 0.03803013227676244, 'dist': 1.9616259199896944, 'src': 'red office chair', 'dst': 'Black ele

In [3]:
# assign objects to rooms, run this after saving rooms.json from the website
assign_rooms_from_json(tracker, "graph_dataset/rooms.json")

In [14]:
plot_scene_graph_over_floorplan_manual(tracker, "graph_dataset/floorplan.png", outfile="graph_dataset/scene_graph.html")

[(0, 21, {'rtype': 'near', 'score': 0.21096969617076589, 'dist': 0.9336244810789158, 'src': 'white desktop printer', 'dst': 'research poster board'}), (0, 7, {'rtype': 'near', 'score': 0.06768075045367923, 'dist': 1.6157721122187716, 'src': 'white desktop printer', 'dst': 'wooden door handle'}), (0, 20, {'rtype': 'near', 'score': 0.11053802418474282, 'dist': 1.3214374461924714, 'src': 'white desktop printer', 'dst': 'wall posters display'}), (0, 1, {'rtype': 'near', 'score': 0.1287320429726476, 'dist': 1.2300133532337398, 'src': 'white desktop printer', 'dst': 'red office chair'}), (0, 19, {'rtype': 'near', 'score': 0.1398265047816346, 'dist': 1.1804117455373975, 'src': 'white desktop printer', 'dst': 'Poster on wall.'}), (1, 34, {'rtype': 'near', 'score': 0.12258124813295253, 'dist': 1.259388752261145, 'src': 'red office chair', 'dst': 'Large white board'}), (1, 37, {'rtype': 'near', 'score': 0.03803013227676244, 'dist': 1.9616259199896944, 'src': 'red office chair', 'dst': 'Black ele

In [4]:
tracker.objects[0].room

'INSITE Lab room 2'

In [26]:
from PIL import Image

def save_obj(obj: MapObject):
    # create oid subfolder in graph_dataset_grayscale/objs
    obj_dir = os.path.join("graph_dataset/objs", str(obj.oid))
    os.makedirs(obj_dir, exist_ok=True)
    
    # save obj point cloud
    pcd_path = os.path.join(obj_dir, "point_cloud.ply")
    o3d.io.write_point_cloud(pcd_path, obj.pcd)
    
    # save obj crops
    img_path = os.path.join(obj_dir, "imgs")
    os.makedirs(img_path, exist_ok=True)
    for i, crop in enumerate(obj.crops):
        img = crop[0]
        img = Image.fromarray(img)
        oid = obj.oid
        img.save(os.path.join(img_path, f"{oid}.png"))
    
    metadata = {
        "clip_ft": obj.clip_ft.tolist(),
        "class_name": obj.class_name,
        "room": obj.room,
        "oid": obj.oid
    }
    
    # save metadata as json in obj directory
    metadata_path = os.path.join(obj_dir, "metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(metadata, f)
    

In [24]:
!rm -rf graph_dataset_grayscale//objs/*

In [27]:
for obj in tracker.objects:
    save_obj(obj)

In [3]:
graph = None
with open("graph_dataset/great_extraction/scene_graph.json", "rb") as f:
    graph = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'graph_dataset/great_extraction/scene_graph.json'

In [5]:
room_file = "graph_dataset/rooms.json"
odo_path = "graph_dataset/raw_scans/iphone/3578aa5730/odometry.csv"

In [6]:
odo_df = pd.read_csv(odo_path)
with open(room_file) as f:
    rooms = json.load(f)["rooms"]
parsed = [(r["room"], parse_room_path(r["path"])) for r in rooms]


In [7]:
odo_df.head()

,timestamp,frame,x,y,z,qx,qy,qz,qw
0,6321.641064,0,0.0,0.0,0.0,0.600268,-0.664773,0.322900,0.305762
1,6321.657732,1,0.0,0.0,0.0,0.600902,-0.665074,0.322655,0.304116
2,6321.674400,2,0.0,0.0,0.0,0.600666,-0.665195,0.322539,0.304441
3,6321.691068,3,0.0,0.0,0.0,0.600144,-0.665090,0.322510,0.305727
4,6321.707735,4,0.0,0.0,0.0,0.598922,-0.665001,0.322512,0.308305


In [8]:
from configs.loader import cfg

detection_model = YOLODetector(cfg)

[YOLO Detector] Loading ./Model/weights/yolov8l-world.pt


In [ ]:
import PIL
BAD_CHARS = {'$', '#', '@', '!', '%', '^', '&', '*', '(', ')', '-', '+', '=', 
             '{', '}', '[', ']', '|', '\\', ':', ';', '"', "'", '<', '>', ',', '.', '?', '/'}

labels2idx = {r["room"]: i for i, r in enumerate(rooms)}
images_db = []
labels = []
detected_objects = []
thresh = 0.5
class_names = ['white desktop printer', 'red office chair', 'Large multi-paned window', 'wooden office shelf', 'white office printer', 'beige air purifier', 'wooden display board', 'wooden door handle', 'Whiteboard with writing', 'Red office chair', 'red office stool', 'red office chair', 'Black flat screen', 'blue office chair', 'academic research poster', 'White rectangular board', 'blue office chair', 'Large white whiteboard', 'information poster images', 'Poster on wall.', 'wall posters display', 'research poster board', 'red office chair', 'wooden door frame', 'White trash can', 'White door frame', 'Gray metal panel', 'wooden office door', 'empty classroom tables', 'empty classroom desks', 'blue trash can', 'Gray plastic trash', 'Large white board', 'Large white board.', 'Large white board', 'Round white table.', 'Large white board', 'Black electronic rack.', 'white office table', 'Black flat screen', 'White table surface', 'white rectangular surface', 'Black flat screen', 'white rolling table', 'white rectangular table', 'White window shade', 'white flat surface', 'black flat screen', 'white table top', 'Large flat screen', 'beige hospital door', 'white folding table', 'White rectangular table', 'white folding table', 'white wooden table', 'black flat screen', 'white table top', 'White table top', 'Large white board', 'white folding table', 'white table top', 'white table chairs', 'White table chair.', 'beige table top', 'white metal board', 'white table chairs', 'White table legs', 'White office table', 'Two white boards. \n## Step 1\nThe input captions describe the object from different views.\n\n## Step 2\nThe captions mention "Two white boards."\n\n## 3\nThe output must be exactly 3 words and describe the object.\n\nThe final answer is: $\\boxed{white boards}$', 'wooden door', 'brown wooden door', 'elevator control panel', 'brown wooden door', 'wooden glass door', 'Wooden bookshelf papers', 'wooden glass door', 'wooden office door', 'brown wooden bookshelf', 'wooden bookshelf unit', 'brown wooden box', "colorful children's poster", 'rainbow painted bench', 'metal payphone box', 'Gray bathroom counter', 'The object unidentifiable.', 'white framed window', 'silver metal table', 'Metal table base.', 'white air conditioning', 'Cream colored object', 'brown water fountain', 'Gray trash can', 'white window frame', 'Large black chalkboard', 'Large black chalkboard', 'Gray plastic chair.', 'black chalkboard surface', 'black chalk board', 'computer server rack', 'Apple computer monitor.', 'computer tower keyboard', 'Black corded phone', 'Black computer monitor', 'Black metal cart.', 'computer gray monitor', 'white wall poster', 'red carpeted stairs', 'white folding table', 'white table top', 'white folding table', 'Large white appliance', 'white folding table', 'White table top', 'Long white table', 'White office table', 'white table desk', 'black chalkboard board', 'black chalkboard wall', 'Long white table.', 'white table top', 'white folding table', 'White folding table', 'Long white table', 'white roller shade', 'white folding table', 'white folding table', 'black chalkboard surface', 'black chalkboard surface', 'wooden glass door', 'Wooden glass door', 'Wooden door panel', 'metal shelving unit', 'Metal storage rack', 'Glass table top.', 'Gray metal table', 'brown rectangular table', 'brown office desk', 'brown wooden table', 'Brown table placemat', 'white folding table', 'There is no central object. \nHowever, based on typical scene understanding, the output could be:\nlarge window frame', 'man in shirt', 'pile of brochures', 'colorful paper materials', 'Dark tile counter', 'dark wooden counter', 'Wooden door frame', 'Wooden glass door', 'Wooden glass door', 'Coca Cola vending', 'black vending machine', 'wooden door panel', 'beige kitchen cabinet', 'Black vending machine', 'Glass door refrigerator', 'vending machine stocked', 'Glass refrigerator door', 'brown wooden door', 'wooden office desk', 'brown wooden dresser', 'brown wooden table', 'white window frame', 'wooden office desk', 'wooden chair cushion', 'wooden brown desk', 'white window frame', 'wooden night stand', 'wooden office desk', 'white window shade', 'White door window', 'white hallway doors', 'glass door frame', 'brown wooden door', 'brown wooden door', 'black office floor', 'Dark brown door', 'White curtain panel', 'beige wall section', 'Wooden door frame', 'window with blinds', 'Gray carpeted walkway', 'Wooden door window', 'water fountain machine', 'metal floor vent', 'white window frame', 'wooden office door', 'blue white walls', 'brown wooden frame', 'Brown wooden door', 'wooden door panel', 'wooden glass door', 'Glass office divider', 'black trash can', 'wooden glass door', 'Wooden glass door', 'glass office door', 'glass partition wall', 'white office wall', 'glass office door', 'empty office space', 'Glass office doors', 'glass office doors', 'white glass door', 'white rectangular board', 'white notice board', 'brown wooden table', 'wooden office table', 'Glass door panel', 'Gray office carpet', 'glass office door', 'Notice board sign', 'glass white board', 'Man in glasses.', 'wooden table chairs', 'Yellow sign board', 'white curtain door', 'brown wooden door', 'brown wooden table', 'white rectangular board', 'glass office door', 'brown wooden door', 'Brown office desk', 'Glass office wall', 'brown wooden door', 'Exit sign panel', 'Wooden glass door', 'Blue leather chair', 'Long black table', 'Long black table', 'colorful book pile', "children's book collection", 'colorful paper brochures']
cls = [n for n in class_names if BAD_CHARS.isdisjoint(n)]
detection_model.model.set_classes(cls)
detection_model.class_names = cls

for i in range(odo_df.shape[0]):
    x, z = odo_df[" x"].iloc[i], odo_df[" z"].iloc[i]
    for room_name, room_path in parsed:
        if inside(x, z, room_path):
            frame_path = f"graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/{odo_df[' frame'].iloc[i]:06d}.png"
            if not os.path.exists(frame_path):
                print(f"Frame path {frame_path} does not exist, skipping.")
                continue
            im_arr = PIL.Image.open(frame_path)
            im_arr = np.array(im_arr)
            bbox, cls_idx, scores = detection_model(im_arr)
            captions = []
            for i in range(len(bbox)):
                if float(scores[i]) < thresh:
                    continue
                captions.append(cls[int(cls_idx[i])])
            detected_objects.append(captions)
            images_db.append(im_arr)
            labels.append(labels2idx[room_name])

Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006413.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006414.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006417.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006418.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006419.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006421.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006422.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006423.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006426.png does not exist, skipping.
Frame path graph_dataset/raw_scans/iphone/3578aa5730/rgb_frames/006427.png does not exist, 

In [8]:
images_db = np.array(images_db)
images_db.shape

(9816, 960, 720, 3)

In [10]:
im_db_set = {
    "images": images_db,
    "images_captions": detected_objects,
    "i_label": labels,
    "label2i": labels2idx
}

with open("./image_db.pkl", 'wb') as f:
    pickle.dump(im_db_set, f)

In [2]:
data = load_spot_data(window_size=5)

100%|██████████| 214/214 [00:09<00:00, 22.37it/s]

[[      1.841       1.841       1.841 ...      22.585      22.544      22.417]
 [     1.8407      1.8407      1.8407 ...      22.585      22.544      22.418]
 [     1.8412      1.8412      1.8412 ...      22.585      22.544      22.417]
 ...
 [     28.579      28.579      28.579 ...      5.1672      5.2041       5.321]
 [      27.24       27.24       27.24 ...      4.0136      4.0452       4.148]
 [     26.024      26.024      26.024 ...      3.1389      3.1609      3.2373]]


In [3]:
len(data["db_images"])

10171

In [4]:
len(data["ts"])

1278

In [4]:
data['ground_truth'][10][1][-1]

8419

In [3]:
import pickle

with open("spot_dataset_w_gt_5m.pkl", "wb") as f:
    pickle.dump(data, f)

In [13]:
np.array(data["ts"]) == 0

array([ True,  True,  True, ..., False, False, False])